# 0.0. Setup

In [1]:
import os
from pathlib import Path

import re
import shutil
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
PUBLIC_DIR = PROJECT_DIR / 'data' / 'public'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 1.0. Preprocess Data

We will use the processed enrollment files the Strategic Management Strand released in the Project Bukas [website](https://www.deped.gov.ph/machine-ready-files/).

In [2]:
!ls "{DATA_DIR}/processed"

'Geolocation of Public Schools_DepEd.xlsx'
 SY_2022_2023_School_Level_Data_on_Official_Enrollment.csv
 SY_2023_2024_School_Level_Data_on_Official_Enrollment.csv
 SY_2024_2025_School_Level_Data_on_Official_Enrollment.csv
 esc_beneficiaries.parquet
 project_bukas_enrollment_2022-23.csv
 project_bukas_enrollment_2023-24.csv
 project_bukas_enrollment_2024-25.csv
 psgc_shapefiles.pkl
 raw_validation_sheets


In [3]:
dirpath = str(PROCESSED_DIR)
pattern = r'^project_bukas'
filenames = [file for file in os.listdir(dirpath) if re.search(pattern, file)]
filepaths = [os.path.join(dirpath, fname) for fname in filenames]
display(filepaths)

['/workspace/project_paaral/data/processed/project_bukas_enrollment_2022-23.csv',
 '/workspace/project_paaral/data/processed/project_bukas_enrollment_2023-24.csv',
 '/workspace/project_paaral/data/processed/project_bukas_enrollment_2024-25.csv']

## 1.1. Clean one

In [4]:
df_csv = pd.read_csv(filepaths[2]) # SY 24-25
print(df_csv.shape)
display(df_csv.head(3))

(60129, 78)


,school_id,school_name,region,division,province,school_district,legislative_district,municipality,barangay,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,kinder_male,kinder_female,g1_male,g1_female,g2_male,g2_female,g3_male,g3_female,g4_male,g4_female,g5_male,g5_female,g6_male,g6_female,esng_male,esng_female,g7_male,g7_female,g8_male,g8_female,g9_male,g9_female,g10_male,g10_female,jhsng_male,jhsng_female,g11_abm_male,g11_abm_female,g11_arts_male,g11_arts_female,g11_gas_male,g11_gas_female,g11_humss_male,g11_humss_female,g11_maritime_male,g11_maritime_female,g11_sports_male,g11_sports_female,g11_stem_male,g11_stem_female,g11_tvl_male,g11_tvl_female,g11_unique_male,g11_unique_female,g12_abm_male,g12_abm_female,g12_arts_male,g12_arts_female,g12_gas_male,g12_gas_female,g12_humss_male,g12_humss_female,g12_maritime_male,g12_maritime_female,g12_sports_male,g12_sports_female,g12_stem_male,g12_stem_female,g12_tvl_male,g12_tvl_female,g12_unique_male,g12_unique_female
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,LIBTONG,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,4,7,3,4,3,3,5,7,5,2,2,2,3,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,100002,Bacarra CES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,SANTA RITA (POB.),Santa Rita,Public,DepEd,Standalone School,True,False,False,22,19,30,25,34,26,17,29,22,29,29,23,34,34,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,100003,Buyon ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,BUYON,NaN,Public,DepEd,Standalone School,True,False,False,7,8,6,7,12,11,9,4,12,5,6,7,10,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
tmp_df = df_csv.copy()
tmp_df['school_id'] = tmp_df['school_id'].astype(str)

# Let's remove PSO schools
mask_pso = tmp_df['region'] != 'PSO'
tmp_df = tmp_df.loc[mask_pso]
df_idxs = tmp_df['school_id'].values
# print(tmp_df.shape)

# Let's aggregate enrollment from kinder to grade 10
grade_lvls_k10 = [
    'enr_kindergarten',
    *['enr_grade_'+str(num) for num in range(1,7)],
    'enr_esng',
    *['enr_grade_'+str(num) for num in range(7,11)],
    'enr_jhsng',
]
# print(grade_lvls_k10)

k10_enr_cols = tmp_df.loc[:, 'kinder_male':'jhsng_female'].columns.tolist()
enr_col_pairs = [k10_enr_cols[r:r+2] for r in range(0, len(k10_enr_cols), 2)]

vals, col_labels = [], []
for i, lvl in enumerate(grade_lvls_k10):
    # print(enr_col_pairs[i])
    col_vals = tmp_df[enr_col_pairs[i]].sum(axis=1)
    col_labels.append(lvl)
    vals.append(col_vals)
    # break

# Let's aggregate enrollment from grade 11 to 12
grade_lvls_k12 = ['enr_grade_11', 'enr_grade_12']
k12_enr_cols = tmp_df.loc[:, 'g11_abm_male':].columns.tolist()
enr_pairs = [k12_enr_cols[:18], k12_enr_cols[18:]]

for i, lvl in enumerate(grade_lvls_k12):
    col_vals = tmp_df[enr_pairs[i]].sum(axis=1)
    col_labels.append(lvl)
    vals.append(col_vals)

df_vals = np.column_stack(vals)
k12_agg_enr = pd.DataFrame(
    data=df_vals,
    columns=col_labels,
    index=df_idxs
)
# print(k12_agg_enr.shape)
# display(k12_agg_enr.head())

# Append aggregated enrollment to original df and drop og enrollment columns
processed_enr = (
    pd.concat(
        [
            tmp_df.set_index('school_id'),
            k12_agg_enr
        ],
        axis=1
    )
    .drop(columns=k10_enr_cols+k12_enr_cols)
)
processed_enr.index.name = 'school_id'

info_cols = processed_enr.loc[:,'school_name':'offers_shs'].columns
df_info = processed_enr[info_cols].copy()

# Affix data types
for col in df_info[info_cols].columns:
    if col in ['offers_es','offers_jhs','offers_shs']:
        processed_enr[col] = processed_enr[col].astype(bool)
    else:
        processed_enr[col] = processed_enr[col].astype(str)

print(processed_enr.shape)
display(processed_enr.head(3))

(60094, 30)


,school_name,region,division,province,school_district,legislative_district,municipality,barangay,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,enr_kindergarten,enr_grade_1,enr_grade_2,enr_grade_3,enr_grade_4,enr_grade_5,enr_grade_6,enr_esng,enr_grade_7,enr_grade_8,enr_grade_9,enr_grade_10,enr_jhsng,enr_grade_11,enr_grade_12
school_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,LIBTONG,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,11,7,6,12,7,4,12,0,0,0,0,0,0,0,0
100002,Bacarra CES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,SANTA RITA (POB.),Santa Rita,Public,DepEd,Standalone School,True,False,False,41,55,60,46,51,52,68,0,0,0,0,0,0,0,0
100003,Buyon ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,BUYON,nan,Public,DepEd,Standalone School,True,False,False,15,13,23,13,17,13,24,0,0,0,0,0,0,0,0


In [6]:
def nan_enrollment(df_processed):
    "Assign NaN to enrollment values based on if the school offers es, jhs, or shs"
    df = df_processed.copy()
    
    # Define column groups for each school level
    es_cols = ['enr_kindergarten', *[f'enr_grade_{i}' for i in range(1, 7)], 'enr_esng']
    jhs_cols = [f'enr_grade_{i}' for i in range(7, 11)] + ['enr_jhsng']
    shs_cols = ['enr_grade_11', 'enr_grade_12']
    
    # Create boolean masks as DataFrames with matching index for proper alignment
    es_mask = pd.concat([df['offers_es']] * len(es_cols), axis=1)
    es_mask.columns = es_cols
    
    jhs_mask = pd.concat([df['offers_jhs']] * len(jhs_cols), axis=1)
    jhs_mask.columns = jhs_cols
    
    shs_mask = pd.concat([df['offers_shs']] * len(shs_cols), axis=1)
    shs_mask.columns = shs_cols
    
    # Vectorized masking: keep value if offers_X is True, else NaN
    df[es_cols] = df[es_cols].where(es_mask)
    df[jhs_cols] = df[jhs_cols].where(jhs_mask)
    df[shs_cols] = df[shs_cols].where(shs_mask)
    
    return df

enr_nan = nan_enrollment(processed_enr)

enr_cols = enr_nan.loc[:,'enr_kindergarten':].columns
df_enrollment = enr_nan[enr_cols].copy()
print(df_enrollment.shape)

(60094, 15)


In [7]:
display(df_info.head())

,school_name,region,division,province,school_district,legislative_district,municipality,barangay,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs
school_id,,,,,,,,,,,,,,,
100001,Apaleng-Libtong ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,LIBTONG,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False
100002,Bacarra CES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,SANTA RITA (POB.),Santa Rita,Public,DepEd,Standalone School,True,False,False
100003,Buyon ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,BUYON,NaN,Public,DepEd,Standalone School,True,False,False
100004,Ganagan Elementary School,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,GANAGAN,"#37 Ganagan,Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False
100005,Macupit ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,MACUPIT,Macupit,Public,DepEd,Standalone School,True,False,False


In [8]:
display(df_enrollment.head())

,enr_kindergarten,enr_grade_1,enr_grade_2,enr_grade_3,enr_grade_4,enr_grade_5,enr_grade_6,enr_esng,enr_grade_7,enr_grade_8,enr_grade_9,enr_grade_10,enr_jhsng,enr_grade_11,enr_grade_12
school_id,,,,,,,,,,,,,,,
100001,11.0,7.0,6.0,12.0,7.0,4.0,12.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100002,41.0,55.0,60.0,46.0,51.0,52.0,68.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100003,15.0,13.0,23.0,13.0,17.0,13.0,24.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100004,12.0,17.0,9.0,10.0,13.0,11.0,13.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100005,11.0,6.0,6.0,5.0,9.0,7.0,8.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.2. Three years

### Class

In [9]:
class preprocess_enrollment:
    def __init__(
        self,
        remove_pso=False,
        verbose=False,
        save_output=False,
        use_psgc=False,
    ):
        self.verbose = verbose
        self.remove_pso = remove_pso
        
        dirpath = str(PROCESSED_DIR) # 'data/processed'
        pattern = r'^project_bukas'
        filenames = [file for file in os.listdir(dirpath) if re.search(pattern, file)]
        self.filepaths = [os.path.join(dirpath, fname) for fname in filenames]

        self.save_output = save_output
        
        self.use_psgc = use_psgc
        self.fpath_psgc = PUBLIC_DIR / 'SY 2024-2025 School Level Database WITH PSGC.xlsx'

        # Make output/ directory, in case none
        if not os.path.exists(self.dirpath_output):
            os.mkdir(self.dirpath_output)

        self.dirpath_output = str(OUTPUT_DIR)

    def vprint(self, print_input):
        if self.verbose:
            print(print_input)
        else:
            pass

    def process_one_school_year(self, df_csv):
        tmp_df = df_csv.copy()
        self.vprint(f"Shape of CSV is {tmp_df.shape}")
        tmp_df['school_id'] = tmp_df['school_id'].astype(str)
        
        # Let's remove PSO schools
        if self.remove_pso:
            mask_pso = tmp_df['region'] != 'PSO'
            tmp_df = tmp_df.loc[mask_pso]
            df_idxs = tmp_df['school_id'].values
        else:
            df_idxs = tmp_df['school_id'].values
        
        # Let's aggregate enrollment from kinder to grade 10
        grade_lvls_k10 = [
            'enr_kindergarten',
            *['enr_grade_'+str(num) for num in range(1,7)],
            'enr_esng',
            *['enr_grade_'+str(num) for num in range(7,11)],
            'enr_jhsng',
        ]
        
        k10_enr_cols = tmp_df.loc[:, 'kinder_male':'jhsng_female'].columns.tolist()
        enr_col_pairs = [k10_enr_cols[r:r+2] for r in range(0, len(k10_enr_cols), 2)]
        
        vals, col_labels = [], []
        for i, lvl in enumerate(grade_lvls_k10):
            col_vals = tmp_df[enr_col_pairs[i]].sum(axis=1)
            col_labels.append(lvl)
            vals.append(col_vals)
        
        # Let's aggregate enrollment from grade 11 to 12
        grade_lvls_k12 = ['enr_grade_11', 'enr_grade_12']
        k12_enr_cols = tmp_df.loc[:, 'g11_abm_male':].columns.tolist()
        enr_pairs = [k12_enr_cols[:18], k12_enr_cols[18:]]
        
        for i, lvl in enumerate(grade_lvls_k12):
            col_vals = tmp_df[enr_pairs[i]].sum(axis=1)
            col_labels.append(lvl)
            vals.append(col_vals)
        
        df_vals = np.column_stack(vals)
        k12_agg_enr = pd.DataFrame(
            data=df_vals,
            columns=col_labels,
            index=df_idxs
        )
        
        # Append aggregated enrollment to original df and drop og enrollment columns
        processed_enr = (
            pd.concat(
                [
                    tmp_df.set_index('school_id'),
                    k12_agg_enr
                ],
                axis=1
            )
            .drop(columns=k10_enr_cols+k12_enr_cols)
        )
        processed_enr.index.name = 'school_id'
        
        info_cols = processed_enr.loc[:,'school_name':'offers_shs'].columns
        
        # Affix data types for school info columns
        for col in processed_enr[info_cols].columns:
            if col in ['offers_es','offers_jhs','offers_shs']:
                processed_enr[col] = processed_enr[col].astype(bool)
            else:
                processed_enr[col] = processed_enr[col].astype(str)

        return processed_enr

    def nan_enrollment(self, df_processed):
        "Assign NaN to enrollment values based on if the school offers es, jhs, or shs"
        df = df_processed.copy()
        
        # Define column groups for each school level
        es_cols = ['enr_kindergarten', *[f'enr_grade_{i}' for i in range(1, 7)], 'enr_esng']
        jhs_cols = [f'enr_grade_{i}' for i in range(7, 11)] + ['enr_jhsng']
        shs_cols = ['enr_grade_11', 'enr_grade_12']
        
        # Create boolean masks as DataFrames with matching index for proper alignment
        es_mask = pd.concat([df['offers_es']] * len(es_cols), axis=1)
        es_mask.columns = es_cols
        
        jhs_mask = pd.concat([df['offers_jhs']] * len(jhs_cols), axis=1)
        jhs_mask.columns = jhs_cols
        
        shs_mask = pd.concat([df['offers_shs']] * len(shs_cols), axis=1)
        shs_mask.columns = shs_cols
        
        # Vectorized masking: keep value if offers_X is True, else NaN
        df[es_cols] = df[es_cols].where(es_mask)
        df[jhs_cols] = df[jhs_cols].where(jhs_mask)
        df[shs_cols] = df[shs_cols].where(shs_mask)
        
        return df

    def process_all_school_years(self):
        out_dict = {}
        dict_sch_ids = {}
        for fpath in self.filepaths:
            pattern_sy = r'enrollment_(.*?).csv'
            sy = re.findall(pattern_sy, fpath)[0]
            sy = sy.replace('-','_')

            # # DEBUG
            # print(f"At filepath: {fpath}\nwith SY: {sy}")

            sy_csv = pd.read_csv(fpath)
            df_sy_processed = self.process_one_school_year(sy_csv)
            df_sy_na = self.nan_enrollment(df_sy_processed)
            df_sy_na['school_year'] = sy

            order_cols = (
                df_sy_na.loc[:,:'offers_shs'].columns.tolist() +
                ['school_year'] + 
                df_sy_na.loc[:,'enr_kindergarten':'enr_grade_12'].columns.tolist()
            )

            out_dict[sy] = df_sy_na[order_cols]
            dict_sch_ids[sy] = df_sy_na.index

        self.dict_sch_ids = dict_sch_ids
        
        return out_dict

    def build_school_tables(self, dict_sys):
        dfs_info, sys = [], []
        dfs_enr = []
        for sy, df_sy in dict_sys.items():
            # information
            df_info = df_sy.loc[:,:'offers_shs']
            dfs_info.append(df_info.reset_index())
            sys.append(sy)

            # enrollment values
            df_enr = df_sy.loc[:,'school_year':]
            dfs_enr.append(df_enr.reset_index())

        # information
        dfs_info = pd.concat(dfs_info)
        dfs_info = dfs_info.drop_duplicates(subset='school_id')

        def is_info_in_sy(school_ids, x):
            if x in school_ids:
                return True
            else:
                return False
        
        # We'd like to tag schools whose information is or is not present in specific school years
        for sy in sys:
            col_name = 'in_enr_' + sy
            school_ids = self.dict_sch_ids.get(sy)
            dfs_info[col_name] = dfs_info['school_id'].apply(
                lambda x: is_info_in_sy(school_ids, x)
            )

        # enrollment values
        dfs_enr = pd.concat(dfs_enr)
        df_melt = dfs_enr.melt(
            id_vars=['school_id','school_year'],
            value_vars=dfs_enr.loc[:,'enr_kindergarten':].columns,
            var_name='enrollment_label',
            value_name='enrollment_count'
        )
        
        df_melt_enr = df_melt.dropna(subset='enrollment_count').copy()

        # # DEBUG
        # debug_df = df_melt_enr[df_melt_enr['school_id'].isin(["300001","300000"])]
        # display(debug_df[debug_df['enrollment_label'] == 'enr_grade_7'])
        
        def tag_grade_level(x):
            return re.findall(r'enr_(.*)', x)[0]
        
        df_melt_enr['grade_level'] = df_melt_enr['enrollment_label'].apply(tag_grade_level)
        
        def tag_education_level(x):
            if re.search(r'kindergarten|_[1-6]$|esng$', x):
                return 'elementary'
            elif re.search(r'_[7-9]$|grade_10|jhsng$', x):
                return 'junior_high_school'
            else:
                return 'senior_high_school'
        
        df_melt_enr['education_level'] = df_melt_enr['enrollment_label'].apply(tag_education_level)

        return dfs_info, df_melt_enr

    def incorporate_old_regions(self):
        self.school_info['old_region'] = self.school_info['region']

        nir_enc = {
            'SIQUIJOR':'Region VII',
            'NEGROS ORIENTAL':'Region VII',
            'NEGROS OCCIDENTAL':'Region VI'
        }

        # mask_pre_nir = self.school_info['province'].isin(nir_enc.keys())
        for province in nir_enc.keys():
            mask = (self.school_info['region'] == 'NIR') & (self.school_info['province'] == province)
            self.school_info.loc[mask, 'old_region'] = nir_enc.get(province)

    def incorporate_psgc(self):
        df_info = self.school_info.copy()
        df_deped_psgc = self.load_preprocess_deped_psgc()

        # Columns that will be replaced in original school information table when use_psgc=True
        rep_cols_a = [
            'region', 'province',
            'municipality', 'barangay',
         ]
        df_info.drop(columns=rep_cols_a, inplace=True)

        mrg_info = df_info.merge(
            df_deped_psgc,
            left_on=['school_id'],
            right_on=['school_id'],
            how='left'
        )
        
        # Rename columns to generally follow DepEd column naming
        col_enc = {
            'psgc_region':'region_psgc',
            'psgc_region_name':'region',
            'psgc_province':'province_psgc',
            'psgc_province_name':'province',
            'psgc_muncipal/city':'municipality_psgc',
            'psgc_muncipal/city_name':'municipality',
            'psgc_barangay':'barangay_psgc',
            'psgc_barangay_name':'barangay',
            'urban/rural_(based_on_2020_cph)':'urban/rural_2020_cph',
            'new_income_classification_(based_on_dof)':'new_income_classification_dof',
        }
        mrg_info.rename(columns=col_enc, inplace=True)
        self.school_info_psgc = mrg_info

        return mrg_info

    def load_preprocess_deped_psgc(self):
        psgc_excel = pd.read_excel(self.fpath_psgc, sheet_name='DB')
        df_tmp = psgc_excel.copy()
        
        header_n = 5
        headers = df_tmp.loc[header_n, :].values
        df_tmp.columns = ['_'.join(header.strip().lower().replace('\n','').split(r' ')) for header in headers]
        df_tmp = df_tmp.iloc[header_n+1:, :]
        
        # Affix data types
        for col in df_tmp.columns:
            df_tmp[col] = df_tmp[col].astype(str)
        
        # Retain relevant columns
        rel_cols = [
            'beis_school_id',
            '(psgc)_region', '(psgc)_region_name', '(psgc)_province',
            '(psgc)_province_name', '(psgc)_muncipal/city',
            '(psgc)_muncipal/city_name', '(psgc)_barangay',
            '(psgc)_barangay_name', 'urban_/_rural(based_on_2020_cph)',
            'new_income_classification_(based_on_dof)'
        ]
        df_tmp = df_tmp[rel_cols]
        
        # Remove duplicating "new_income_classification_(based_on_dof)"
        df_tmp = df_tmp.iloc[:, :-1]
        
        # Clean header (e.g., remove parenthesis)
        headers_clean = (
            [col.replace('(','').replace(')','') for col in df_tmp.loc[:, :'(psgc)_barangay_name'].columns]
            + ['urban_/_rural(based_on_2020_cph)', 'new_income_classification_(based_on_dof)']
        )
        # print(headers_clean)
        df_tmp.columns = headers_clean
        
        rename_cols = {
            'beis_school_id':'school_id',
            'urban_/_rural(based_on_2020_cph)':'urban/rural_(based_on_2020_cph)',
        }
        df_tmp.rename(columns=rename_cols, inplace=True)

        return df_tmp

    def run(self):
        self.dict_dfs = self.process_all_school_years()
        self.school_info, self.school_enr = self.build_school_tables(self.dict_dfs)
        self.incorporate_old_regions()

        if self.use_psgc:
            self.incorporate_psgc()
        
        if self.save_output:
            self.save()

    def remove_if_exists(self, filepath):
        if os.path.exists(filepath):
            if os.path.isdir(filepath):
                shutil.rmtree(filepath)
                print(f"Deleted directory with path {filepath}")
            else:
                os.remove(filepath)
                print(f"Deleted file with path {filepath}")
    
    def save(self):
        self.vprint(f"Saving processed enrollment tables in output folder...")
        fname = 'processed_project_bukas_school_information.parquet'
        fpath = os.path.join(self.dirpath_output, fname)
        
        self.remove_if_exists(fpath)
        self.school_info.to_parquet(fpath)

        fname = 'processed_project_bukas_school_enrollment.parquet'
        fpath = os.path.join(self.dirpath_output, fname)
        self.remove_if_exists(fpath)
        self.school_enr.to_parquet(fpath, partition_cols='school_year')

        if isinstance(self.school_info_psgc, pd.DataFrame):
            self.vprint(f"Saving school information with PSGC in output folder...")
            fname = 'processed_project_bukas_school_information_psgc.parquet'
            fpath = os.path.join(self.dirpath_output, fname)
            self.remove_if_exists(fpath)
            self.school_info_psgc.to_parquet(fpath)
        
        self.vprint(f"Saving complete!")

### Implem

In [10]:
p_enr = preprocess_enrollment(
    remove_pso=False,
    verbose=True,
    save_output=False,
    use_psgc=False
)

In [11]:
dict_enr = p_enr.run()

Shape of CSV is (60137, 78)
Shape of CSV is (60167, 78)
Shape of CSV is (60129, 78)


In [12]:
tmp_enr = p_enr.school_enr.copy()
print(tmp_enr.shape)
display(tmp_enr.head(1))

(1506312, 6)


,school_id,school_year,enrollment_label,enrollment_count,grade_level,education_level
0,100001,2022_23,enr_kindergarten,6.0,kindergarten,elementary


In [13]:
mask_sy = tmp_enr['school_year'] == '2023_24'
pvt_tmp = tmp_enr[mask_sy].pivot_table(
    index='school_id',
    columns='enrollment_label',
    values='enrollment_count',
    aggfunc='sum',
)
display(pvt_tmp.head(2))

enrollment_label,enr_esng,enr_grade_1,enr_grade_10,enr_grade_11,enr_grade_12,enr_grade_2,enr_grade_3,enr_grade_4,enr_grade_5,enr_grade_6,enr_grade_7,enr_grade_8,enr_grade_9,enr_jhsng,enr_kindergarten
school_id,,,,,,,,,,,,,,,
100000,0.0,54.0,NaN,NaN,NaN,31.0,41.0,36.0,38.0,39.0,NaN,NaN,NaN,NaN,37.0
100001,0.0,5.0,NaN,NaN,NaN,12.0,6.0,4.0,11.0,11.0,NaN,NaN,NaN,NaN,9.0


In [14]:
%%time
# Run this cell if you want to save a version of school information where
# PSA's PSGC replaces DepEd's usual naming of region, provinces, etc.
sch_info_psgc = p_enr.incorporate_psgc()
print(sch_info_psgc.shape)
display(sch_info_psgc.head(3))

(61442, 26)


,school_id,school_name,division,school_district,legislative_district,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region,region_psgc,region,province_psgc,province,municipality_psgc,municipality,barangay_psgc,barangay,urban/rural_2020_cph,new_income_classification_dof
0,100001,Apaleng-Libtong ES,Ilocos Norte,Bacarra I,1st District,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802013,Libtong,R,2nd
1,100002,Bacarra CES,Ilocos Norte,Bacarra I,1st District,Santa Rita,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802046,Santa Rita,R,2nd
2,100003,Buyon ES,Ilocos Norte,Bacarra I,1st District,nan,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802002,Buyon,R,2nd


CPU times: user 7.95 s, sys: 162 ms, total: 8.12 s
Wall time: 8.66 s


In [15]:
p_enr.school_info_psgc.head(3)

,school_id,school_name,division,school_district,legislative_district,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region,region_psgc,region,province_psgc,province,municipality_psgc,municipality,barangay_psgc,barangay,urban/rural_2020_cph,new_income_classification_dof
0,100001,Apaleng-Libtong ES,Ilocos Norte,Bacarra I,1st District,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802013,Libtong,R,2nd
1,100002,Bacarra CES,Ilocos Norte,Bacarra I,1st District,Santa Rita,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802046,Santa Rita,R,2nd
2,100003,Buyon ES,Ilocos Norte,Bacarra I,1st District,nan,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802002,Buyon,R,2nd


In [16]:
# This cell was made for 1.4. notebook to check LRNs issued from PSOs
p_enr.school_info_psgc[p_enr.school_info_psgc['school_id'] == '700005']

,school_id,school_name,division,school_district,legislative_district,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region,region_psgc,region,province_psgc,province,municipality_psgc,municipality,barangay_psgc,barangay,urban/rural_2020_cph,new_income_classification_dof
60106,700005,Al Taj International School,Kingdom of Saudi Arabia,Riyadh,PSO,"Johaina St. Box 15263, Riyadh 11444",PSO,PSO,Standalone School,True,True,True,True,True,True,PSO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Save output

In [27]:
p_enr.save()

Saving processed enrollment tables in output folder...
Deleted file with path /workspace/project_paaral/output/processed_project_bukas_school_information.parquet
Deleted directory with path /workspace/project_paaral/output/processed_project_bukas_school_enrollment.parquet
Saving school information with PSGC in output folder...
Deleted file with path /workspace/project_paaral/output/processed_project_bukas_school_information_psgc.parquet
Saving complete!


### Load output

In [17]:
fpath = OUTPUT_DIR / 'processed_project_bukas_school_information.parquet'
df_enr_info = pd.read_parquet(fpath)
print(df_enr_info.shape)
display(df_enr_info.head())

(61442, 20)


,school_id,school_name,region,division,province,school_district,legislative_district,municipality,barangay,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,LIBTONG,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I
1,100002,Bacarra CES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,SANTA RITA (POB.),Santa Rita,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I
2,100003,Buyon ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,BUYON,nan,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I
3,100004,Ganagan Elementary School,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,GANAGAN,"#37 Ganagan,Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I
4,100005,Macupit ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,MACUPIT,Macupit,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I


In [18]:
df_enr_info['old_region'].unique()

array(['Region I', 'Region II', 'Region III', 'Region IV-A', 'MIMAROPA',
       'Region V', 'Region VI', 'Region VII', 'Region VIII', 'Region IX',
       'Region X', 'Region XI', 'Region XII', 'CARAGA', 'BARMM', 'CAR',
       'NCR', 'PSO'], dtype=object)

In [19]:
fpath = OUTPUT_DIR / 'processed_project_bukas_school_information_psgc.parquet'
df_enr_info_psgc = pd.read_parquet(fpath, engine='fastparquet')
print(df_enr_info_psgc.shape)
display(df_enr_info_psgc.head())

(61442, 26)


,school_id,school_name,division,school_district,legislative_district,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region,region_psgc,region,province_psgc,province,municipality_psgc,municipality,barangay_psgc,barangay,urban/rural_2020_cph,new_income_classification_dof
0,100001,Apaleng-Libtong ES,Ilocos Norte,Bacarra I,1st District,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802013,Libtong,R,2nd
1,100002,Bacarra CES,Ilocos Norte,Bacarra I,1st District,Santa Rita,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802046,Santa Rita,R,2nd
2,100003,Buyon ES,Ilocos Norte,Bacarra I,1st District,nan,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802002,Buyon,R,2nd
3,100004,Ganagan Elementary School,Ilocos Norte,Bacarra I,1st District,"#37 Ganagan,Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802012,Ganagan,R,2nd
4,100005,Macupit ES,Ilocos Norte,Bacarra I,1st District,Macupit,Public,DepEd,Standalone School,True,False,False,True,True,True,Region I,100000000,Region I (Ilocos Region),102800000,Ilocos Norte,102802000,Bacarra,102802014,Macupit,R,2nd


In [20]:
fpath = OUTPUT_DIR / 'processed_project_bukas_school_enrollment.parquet'
df_enr_enr = pd.read_parquet(fpath)
print(df_enr_enr.shape)
display(df_enr_enr.head())

(1506312, 6)


,school_id,enrollment_label,enrollment_count,grade_level,education_level,school_year
0,100001,enr_kindergarten,6.0,kindergarten,elementary,2022_23
1,100002,enr_kindergarten,56.0,kindergarten,elementary,2022_23
2,100003,enr_kindergarten,23.0,kindergarten,elementary,2022_23
3,100004,enr_kindergarten,10.0,kindergarten,elementary,2022_23
4,100005,enr_kindergarten,7.0,kindergarten,elementary,2022_23


In [21]:
df_enr_enr['grade_level'].unique()

array(['kindergarten', 'grade_1', 'grade_2', 'grade_3', 'grade_4',
       'grade_5', 'grade_6', 'esng', 'grade_7', 'grade_8', 'grade_9',
       'grade_10', 'jhsng', 'grade_11', 'grade_12'], dtype=object)

In [22]:
# # Save a SY 23-24 version that is pivotted & from G6-G7 only
# dirpath = OUTPUT_DIR / 'scratch'
# fname = 'processed_sy2324_enrollment.parquet'
# outpath = os.path.join(dirpath, fname)

# if not os.path.exists(dirpath):
#     os.mkdir(dirpath)

# mask = (
#     (df_enr_enr['school_year'].isin(['2023_24']))
#     & (df_enr_enr['grade_level'].isin(['grade_6','grade_7']))
# )
# df_enr_enr = df_enr_enr.loc[mask].drop(columns=['school_year','grade_level'])

# df_enr_enr.to_parquet(outpath)

## 1.3. [IGNORE] SHS Offerings

In [23]:
# from modules import enrollment_processor as ep

In [24]:
# # Initialize processor
# processor_enrollment = ep.EnrollmentDataProcessor()

In [25]:
# %%time
# # Process data
# long_data_enrollment = processor_enrollment.process()

# # Extract SHS offerings
# shs_offerings = processor_enrollment.extract_shs_offerings()

In [26]:
# # Process a little bit
# shs_offerings = shs_offerings.reset_index()
# shs_offerings.rename(columns={'school_id_processed':'school_id'}, inplace=True)
# shs_offerings.columns.name = ''

In [27]:
# display(shs_offerings.head())

### Save

In [28]:
# fname = 'processed_shs_offerings.parquet'
# outpath = os.path.join(OUTPUT_DIR, fname)
# shs_offerings.to_parquet(outpath)